## Part 1: Load Documents & Execute Reranking Model

In [1]:
# %pip install -U pinecone==6.0.1 pinecone-notebooks

In [2]:
# import os
# api_key = "pcsk_4Zssw4_82isLTXuMe7EpTV9USC2R1BhFGnPGfEnZ629oVC2kKvW63YbEsYDS1pEtNLU1LM"
# if not os.environ.get(api_key):
#     from pinecone_notebooks.colab import Authenticate
#     Authenticate()

In [3]:
# from pinecone import Pinecone
# api_key = os.environ.get(api_key)
# pc = Pinecone(api_key=api_key)

import os
from pinecone import Pinecone

api_key = "pcsk_4Zssw4_82isLTXuMe7EpTV9USC2R1BhFGnPGfEnZ629oVC2kKvW63YbEsYDS1pEtNLU1LM"
pc = Pinecone(api_key=api_key)

c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
query = "Tell me about Apple's products"
documents = [
    "Apples are a popular fruit that grow on trees and come in varieties like Gala, Fuji, and Granny Smith.",  # Add a document about apple fruit
    "Apple Inc. designs and sells products such as the iPhone, iPad, MacBook, and Apple Watch.",  # Add a document about Apple company products
    "Bananas are rich in potassium and are one of the most widely consumed fruits in the world.",  # Add another fruit-related document
    "Microsoft is a technology company known for Windows, Office, and the Xbox gaming console.",  # Add another company-related document
    "Oranges are citrus fruits packed with vitamin C and are often juiced for breakfast."  # Add one more document (your choice)
]

In [5]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

In [9]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1} {m.score} {m.document['text']}")

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
1 0.9393595 Apple Inc. designs and sells products such as the iPhone, iPad, MacBook, and Apple Watch.
2 0.025419652 Apples are a popular fruit that grow on trees and come in varieties like Gala, Fuji, and Granny Smith.
3 0.0005571117 Microsoft is a technology company known for Windows, Office, and the Xbox gaming console.


## Part 2: Setup a Serverless Index for Medical Notes

In [10]:
!pip install pandas torch transformers

In [13]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'

In [14]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-pvs1rqa.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

## Part 3: Load the Sample Data

In [16]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/refs/heads/main/Week%208/W8D2/data.json" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

In [17]:
# Show head of the DataFrame
print("Data shape:", df.shape)
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## Part 4: Upsert Data into the Index

In [18]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame

sending upsert requests: 100%|██████████| 100/100 [00:02<00:00, 35.48it/s]


{'upserted_count': 100}

In [19]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0 # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

## Part 5: Query & Embedding Function

In [21]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=0) # Which dimension to average?
    return embedding

In [22]:
# Build a query to search
question = "What are the symptoms of pneumonia?" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=10, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Part 6: Display & Rerank Clinical Notes

In [23]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

Question: 'What are the symptoms of pneumonia?'

Results:
   1. ID: P063
 Score: 0.58763063
 Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   2. ID: P005
 Score: 0.443241924
 Metadata: {'preventive_care': 'flu vaccination', 'vital_signs': 'normal'}

   3. ID: P003
 Score: 0.39198181
 Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   4. ID: P015
 Score: 0.358422101
 Metadata: {'diagnosis': 'sinusitis', 'symptoms': 'sinus pressure, congestion', 'treatment': 'antibiotics'}

   5. ID: P039
 Score: 0.319629729
 Metadata: {'diagnosis': 'ear infection', 'symptoms': 'ear pain', 'treatment': 'antibiotics'}

   6. ID: P087
 Score: 0.319629729
 Metadata: {'diagnosis': 'ear infection', 'symptoms': 'ear pain', 'treatment': 'antibiotics'}

   7. ID: P001
 Score: 0.307611853
 Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   8. ID: P027
 Score: 0.303602755
 Metadata: {'diagnosis': 's

In [24]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

In [25]:
# Define a more specific query for reranking
refined_query = "What are the early symptoms of pneumonia in adults?" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)

In [27]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}') # What attribute contains the reranking score?
        print(f' Reranking Field: {match.document["reranking_field"]}') # What contains the searchable field?
        print('')
show_reranked_results(refined_query, reranked_results.data) # What attribute contains the results?

Question: 'What are the early symptoms of pneumonia in adults?'

Reranked Results:
   1. ID: P063
 Score: 0.3671568
 Reranking Field: diagnosis: pneumonia; symptoms: cough, fever; treatment: antibiotics

   2. ID: P027
 Score: 0.045606945
 Reranking Field: diagnosis: strep throat; symptoms: sore throat, fever; treatment: antibiotics

   3. ID: P099
 Score: 0.030502053
 Reranking Field: diagnosis: flu; symptoms: fever, body aches; treatment: symptomatic



In [28]:
# Delete the index to save resources
pc.delete_index(name=index_name)